In [1]:
from torchvision.datasets import FashionMNIST

from dataeval_flow import PipelineConfig, run_tasks
from dataeval_flow.config import (
    DatasetProtocolConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import BoVWExtractorConfig
from dataeval_flow.workflows.data_cleaning import DataCleaningConfig

# 1. Create the torchvision dataset without transforms. The adapter handles conversion.
tv_dataset = FashionMNIST(root="./data", train=True, download=True)

In [2]:
# 2. Build the pipeline configuration with a subset for faster execution.
datasets = [DatasetProtocolConfig(name="fmnist-train", format="torchvision", dataset=tv_dataset)]
# A bare Limit takes the first N samples in storage order.
# Shuffle first so the subset represents the entire dataset.
views = [
    ViewConfig(
        name="sample500",
        operations=[
            ViewOperation(type="Shuffle", params={"seed": 0}),
            ViewOperation(type="Limit", params={"size": 500}),
        ],
    )
]
sources = [SourceConfig(name="fmnist-src", dataset="fmnist-train", view="sample500")]
extractors = [BoVWExtractorConfig(name="bovw", vocab_size=512, batch_size=64)]

workflows = [
    DataCleaningConfig(
        name="adaptive_clean",
        outlier_method="adaptive",
        outlier_threshold=3.5,
        outlier_flags=["dimension", "pixel", "visual"],
    )
]
tasks = [
    TaskConfig(
        name="fmnist-clean",
        workflow="adaptive_clean",
        sources="fmnist-src",
        extractor="bovw",
    )
]

config = PipelineConfig(
    datasets=datasets,
    views=views,
    sources=sources,
    extractors=extractors,
    workflows=workflows,
    tasks=tasks,
)

In [3]:
# 3. Run, then read the findings and the cleaned dataset
results = run_tasks(config)
result = results["fmnist-clean"]

for finding in result.findings:
    print(f"{finding.severity:<8} {finding.title:<20} {finding.brief}")

clean = result.steps["clean"].output
print(f"\nImages kept: {len(clean)} of {len(result.sources['fmnist-src'])}")

info     Image Outliers       6 images (1.2%)
info     Classwise Outliers   worst: T-shirt/top (3.2%), 1/5 classes over 3.0%
info     Duplicates           0 exact (0.0%), 9 near (1.8%)
info     Label Distribution   10 classes, 500 items, imbalance 1.6:1

Images kept: 488 of 500


In [4]:
DatasetProtocolConfig(
    name="fmnist-train",
    format="torchvision",
    dataset=tv_dataset,
    version="2",  # bump this when the underlying data changes
)

DatasetProtocolConfig(name='fmnist-train', format='torchvision', dataset=Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train, version='2')